# Notebook to demo writing a FileGDB.

Make sure to install `gdal > 3.6`.

References:

- https://pcjericks.github.io/py-gdalogr-cookbook/vector_layers.html
- https://gdal.org/user/virtual_file_systems.html
- https://gis.stackexchange.com/questions/201831/how-to-efficiently-access-files-with-gdal-from-an-s3-bucket-using-vsis3

In [ ]:
import geofunctions as G
import pyspark.sql.functions as F
from osgeo import ogr, osr

### Get the driver.

In [ ]:
driver = ogr.GetDriverByName('OpenFileGDB')
if driver is None:
    print("No support for OpenFileGDB")

### Create points in WKB format.

In [ ]:
rows = (spark
        .range(10_000)
        .withColumn("lon", F.rand() * 360 + 180)
        .withColumn("lat", F.rand() * 180 - 90)
        .withColumn("wkb", G.st_point("lon", "lat"))
        .collect()
        )

### Create the FileGDB.

In [ ]:
ds = driver.CreateDataSource("test.gdb")

### Create the layer.   

In [ ]:
srs = osr.SpatialReference()
srs.ImportFromEPSG(4326)

layer = ds.CreateLayer("Test", srs=srs, geom_type=ogr.wkbPoint)

### Add the fields. 

In [ ]:
idField = ogr.FieldDefn("id", ogr.OFTInteger)
layer.CreateField(idField)

### Get the layer definition.

In [ ]:
featureDefn = layer.GetLayerDefn()

### Add the features.

In [ ]:
for row in rows:
    feat = ogr.Feature(featureDefn)
    geom = ogr.CreateGeometryFromWkb(row.wkb)
    feat.SetGeometry(geom)
    feat.SetField("id", row.id)
    layer.CreateFeature(feat)
    feat = None
    geom = None

### Close the layer and dataset.

In [ ]:
layer = None
ds = None

### Check if we can read back the FileGDB using PySpark.

In [ ]:
df = spark.read.format("gdb").options(path="test.gdb", name="Test").load()

In [ ]:
df.printSchema()

In [ ]:
df.select("Shape.x", "Shape.y", "id").show(truncate=False)